In [69]:
from spiral import Spiral

sp = Spiral()
project = sp.project("external-805943")

experiments_table = project.table("experiments")
eye_trackers_table = project.table("eye-trackers")
responses_table = project.table("responses")
screens_table = project.table("screens")

In [70]:
eye_trackers_table.schema()

Schema({data_key=utf8?, time=f32?, x=f32?, y=f32?, pupil=f32?})

In [71]:
responses_table.schema()

Schema({data_key=utf8?, time=f32?, signals=list(f32?)?})

In [72]:
screens_table.schema()

Schema({data_key=utf8?, time=f32?, frame=binary?})

In [73]:
# Scan is a list of projections.
# Most commonly used projections are column sections.
# But projections that transform column, like slicing a list column, are also possible.
experiment = sp.scan(
    experiments_table[[
        "*",
        "config",
        # * means select columns but do not recurse into sub-column groups.
        "eye_tracker.*",
    ]],
    # Pack result of responses selection back into "responses" column.
    {
        # Customize selection of responses further to drop some sequences we don't need.
        "responses": experiments_table["responses"].select(exclude=["means", "stds"])
    },
    where=experiments_table["data_key"] == "35_3832003544063_1_V1"
).to_table().to_pylist()[0]

experiment

{'data_key': '35_3832003544063_1_V1',
 'config': {'database': {'host': 'at-database3.stanford.edu',
   'password': '',
   'schema_name': 'enigma_acq',
   'user': ''},
  'export': {'compute_report': True,
   'default_fs': 30000,
   'export_suffix': 'permissive_stability_criteria',
   'output_base_dir': '/mnt/stor02/enigma/modeling_pipeline/export/goliath_malachi_v0_3_0/',
   'overwrite': False,
   'target_sampling_rate': 30,
   'use_functional_criteria': False,
   'use_regularity_criteria': False,
   'use_stability_criteria': True,
   'quality_thresholds': {'good_spikes_fraction': 0.3,
    'loo_slope': 0.1,
    'presence_ratio': 0.8,
    'variance_explained': 0.1}},
  'gaze': {'behavior_mnt_dir': ['/mnt/scratch14/', '/mnt/scratch12/'],
   'blink_acc_threshold': 150000.0,
   'blink_detection': False},
  'processing': {'n_jobs': 10},
  'screen': {'dtype': 'float16',
   'frame_t_max': 0.04,
   'frame_t_min': 0.026,
   'frames_in_split': 50,
   'gray_value': 128.0,
   'max_delta_pauses': 0.

In [74]:
import pyarrow as pa

def constant_array(value, length, pa_type):
    """
    Create a constant array in PyArrow with optimal memory efficiency.

    Parameters:
    -----------
    value : any
        The constant value to repeat
    length : int
        The length of the array
    pa_type : pa.DataType
        The PyArrow data type

    Returns:
    --------
    pa.Array
        A constant array with the specified value repeated
    """
    # Create a scalar with the specified type
    scalar = pa.scalar(value, type=pa_type)

    # Use repeat to create a constant array efficiently
    # This uses run-length encoding internally for maximum efficiency
    return pa.repeat(scalar, length)

In [75]:
import numpy as np

responses_time = np.linspace(experiment["responses"]["start_time"], experiment["responses"]["end_time"], experiment["responses"]["n_timestamps"], endpoint=False, dtype=np.float32)
eye_tracker_time = np.linspace(experiment["eye_tracker"]["start_time"], experiment["eye_tracker"]["end_time"], experiment["eye_tracker"]["n_timestamps"], endpoint=False, dtype=np.float32)

In [76]:
signals_scan = sp.scan(responses_table.select("signals"))

# Result schema.
signals_scan.schema

Schema({signals=list(f32?)?})

In [77]:
# Scan can repeatedly be used to fetch results with a different key tables.
signals = signals_scan.to_table(key_table=pa.table({
    "data_key": constant_array("35_3832003544063_1_V1", 25_000, pa.string()),
    "time": pa.array(responses_time[100_000:125_000], type=pa.float32()),
}))

signals

pyarrow.Table
signals: list<item: float>
  child 0, item: float
----
signals: [[[0,0,0,0,0,...,0,0,0,0,0],[0,0,0,0,0,...,0,0,0,0,0],...,[0,0,0,0,0,...,1,0,0,0,0],[0,0,0,0,1,...,0,0,0,0,0]],[[0,1,0,0,0,...,0,0,0,0,0],[0,0,0,0,0,...,0,0,0,0,0],...,[0,0,0,0,0,...,0,0,0,0,0],[0,1,0,0,0,...,0,0,0,0,0]]]

In [78]:
eye_tracker_scan = sp.scan(eye_trackers_table.select("x", "y", "pupil"), where=eye_trackers_table["data_key"] == "35_3832003544063_1_V1")

eye_tracker_scan.schema

Schema({x=f32?, y=f32?, pupil=f32?})

In [79]:
# Get all eye tracker data for an experiment (filter applied on scan).
eye_tracker_scan.to_table()

pyarrow.Table
x: float
y: float
pupil: float
----
x: [[189.43404,189.36218,189.4979,189.54979,189.44202,...,-114.86065,-114.760864,-114.768845,-114.53334,-114.8367]]
y: [[-79.00198,-78.95872,-78.802315,-78.97203,-79.24491,...,-35.833954,-35.784035,-35.43129,-35.697514,-35.391357]]
pupil: [[-99497,-99520,-99491,-99535,-99525,...,-99112,-99118,-99081,-99059,-99081]]